# SoftMeta Chatterbox TTS Server v1.6.1

Fresh Chatterbox Turbo reset for reliable long-form narration.

- **Chatterbox Turbo only** with the official Turbo sampling defaults.
- No Auto Emotion, pronunciation rewriting, Senior Clear Speech, pacing, Production QC, ASR, speaker verification, retry/rescue loops, Prosody processing, captions or video-master generation.
- Long scripts are split only at sentence-safe boundaries into **maximum 300-character Turbo calls** so the original wording is passed through without a narration rewrite layer.
- The only final audio processing is a **loudness boost** to keep the stronger output volume used in recent builds.
- The server stays under a keep-alive supervisor after launch. If the child server exits unexpectedly, it is restarted automatically while the Colab runtime itself is still alive.
- The Audio Studio can manually request **Disconnect Colab** when work is finished. After a runtime is disconnected, reconnect from the Colab page's Connect button.
- L4 GPU is requested by notebook metadata. Actual allocation and maximum session duration are controlled by Google Colab.


In [ ]:
# Setup choices
# Normally keep FORCE_REINSTALL False so the existing L4 environment is reused.

FORCE_REINSTALL = False  # @param {type:"boolean"}

import os
os.environ["SOFTMETA_FORCE_REINSTALL"] = "1" if FORCE_REINSTALL else "0"
os.environ["MPLBACKEND"] = "Agg"
print("Force reinstall:", FORCE_REINSTALL)


## Install the fresh Chatterbox Turbo server

This cell reuses the existing environment when possible and installs only the packages needed by the fresh Turbo server. Heavy ASR and speaker-verification packages are not installed.


In [ ]:
%%bash
set -euo pipefail

export PIP_CACHE_DIR=/content/.cache/pip
missing_system=0
for command_name in ffmpeg git curl lsof; do
  command -v "$command_name" >/dev/null 2>&1 || missing_system=1
done
if [[ "$missing_system" == "1" ]]; then
  apt-get update -qq
  apt-get install -y -qq ffmpeg libsndfile1 git curl ca-certificates lsof build-essential
else
  echo "System packages are already available."
fi

echo "GPU assigned by Colab:"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

mkdir -p /content/bin /content/.cache/pip
MICROMAMBA=/content/bin/micromamba
MICROMAMBA_VERSION=2.6.2-1
MICROMAMBA_URL="https://github.com/mamba-org/micromamba-releases/releases/download/${MICROMAMBA_VERSION}/micromamba-linux-64"

if [[ ! -x "$MICROMAMBA" ]]; then
  curl --fail --location --retry 5 --retry-delay 2 --retry-all-errors \
    --connect-timeout 30 "$MICROMAMBA_URL" --output "$MICROMAMBA"
  chmod +x "$MICROMAMBA"
fi
"$MICROMAMBA" --version

has_env() {
  "$MICROMAMBA" env list | awk '{print $1}' | grep -qx "$1"
}

if [[ "${SOFTMETA_FORCE_REINSTALL:-0}" == "1" ]] && has_env sm311; then
  "$MICROMAMBA" env remove -n sm311 -y || true
fi
if ! has_env sm311; then
  "$MICROMAMBA" create -y -n sm311 -c conda-forge python=3.11 pip
fi

cd /content

if [[ -d chatterbox-v2/.git ]]; then
  git -C chatterbox-v2 fetch --depth 1 origin tag v0.2.1
  git -C chatterbox-v2 checkout --detach v0.2.1
  git -C chatterbox-v2 reset --hard v0.2.1
else
  rm -rf chatterbox-v2
  git clone --branch v0.2.1 --depth 1 https://github.com/soft-meta/chatterbox-v2.git
fi

SERVER_REPO=/content/SoftMeta-Chatterbox-Repository
if [[ -d "$SERVER_REPO/.git" ]]; then
  git -C "$SERVER_REPO" fetch --depth 1 origin main
  git -C "$SERVER_REPO" checkout main
  git -C "$SERVER_REPO" reset --hard origin/main
else
  git clone --branch main --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git "$SERVER_REPO"
fi

# Prefer a complete repository root. If an older nested upload exists, fall back to
# the newest complete server folder found recursively.
PROJECT_SOURCE=$(python3 - <<'PYPROJECT'
import re
from pathlib import Path

repo = Path('/content/SoftMeta-Chatterbox-Repository')

def complete(project: Path) -> bool:
    return all((project / rel).exists() for rel in (
        'server.py', 'requirements-colab.txt', 'start.py', 'ui/index.html'
    ))

def version_of(project: Path):
    text = (project / 'server.py').read_text(encoding='utf-8', errors='replace')
    match = re.search(r'APP_VERSION\s*=\s*"([0-9.]+)"', text)
    return tuple(int(part) for part in match.group(1).split('.')) if match else (0,)

candidates = []
if complete(repo):
    candidates.append((version_of(repo), 1, 0, repo))
for server_file in repo.glob('**/server.py'):
    project = server_file.parent
    if project == repo or not complete(project):
        continue
    depth = len(project.relative_to(repo).parts)
    candidates.append((version_of(project), 0, -depth, project))

if not candidates:
    print('Repository contents:', file=__import__('sys').stderr)
    for path in sorted(repo.rglob('*'))[:120]:
        print(' -', path.relative_to(repo), file=__import__('sys').stderr)
    raise SystemExit('No complete SoftMeta server folder was found in the GitHub repository.')

print(max(candidates)[3])
PYPROJECT
)
echo "Selected server source: $PROJECT_SOURCE"
printf '%s\n' "$PROJECT_SOURCE" > /content/softmeta_project_path.txt

MM=/content/bin/micromamba

if ! "$MM" run -n sm311 python - <<'PYCHECK' >/dev/null 2>&1
import torch
import torchaudio
import chatterbox
import perth
PYCHECK
then
  "$MM" run -n sm311 python -m pip install -U pip wheel
  "$MM" run -n sm311 python -m pip install "setuptools==80.9.0"
  "$MM" run -n sm311 python -m pip install \
    --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0 torchaudio==2.6.0
  "$MM" run -n sm311 python -m pip install --no-cache-dir chatterbox-tts==0.1.7
fi

"$MM" run -n sm311 python -m pip install --no-deps -e /content/chatterbox-v2
"$MM" run -n sm311 python -m pip install -r "$PROJECT_SOURCE/requirements-colab.txt"
"$MM" run -n sm311 python -m pip install "setuptools==80.9.0"

"$MM" run -n sm311 python -m compileall -q "$PROJECT_SOURCE"
SOFTMETA_PROJECT_SOURCE="$PROJECT_SOURCE" "$MM" run -n sm311 python - <<'PYIMPORT'
import os
import sys
sys.path.insert(0, os.environ['SOFTMETA_PROJECT_SOURCE'])
import server
assert server.APP_VERSION == '1.6.1', server.APP_VERSION
print('Server source import check passed:', server.APP_VERSION)
PYIMPORT

echo "Fresh Chatterbox Turbo environment is ready."


## Verify the Turbo runtime

This verifies the GPU and the core Chatterbox packages only. There is no ASR or speaker-verification runtime in v1.6.1.


In [ ]:
%%bash
set -euo pipefail
export MPLBACKEND=Agg
MM=/content/bin/micromamba

"$MM" run -n sm311 python - <<'PYMAIN'
import sys
from importlib.metadata import version
import torch
import torchaudio
import chatterbox
import perth
from softmeta_chatterbox import SoftMetaChatterboxEngine

print("Fresh Chatterbox Turbo environment")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("Transformers:", version("transformers"))
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable. Select a GPU runtime.")
print("GPU:", torch.cuda.get_device_name(0))
print("Official Chatterbox package:", chatterbox.__file__)
print("PerTh watermarker callable:", callable(getattr(perth, "PerthImplicitWatermarker", None)))
print("SoftMeta adapter:", SoftMetaChatterboxEngine.__name__)
PYMAIN

echo "Core Turbo runtime verification finished."


## Start SoftMeta Audio Studio

Leave `KEEP_RUNTIME_ACTIVE` enabled for normal use. The cell remains active and watches the child server. If the child server exits unexpectedly while Colab is still alive, it is restarted automatically.

When all work is finished, use **Disconnect Colab** in the Audio Studio. If you only want to stop the server without releasing the Colab runtime, stop this notebook cell manually.


In [ ]:
KEEP_RUNTIME_ACTIVE = True  # @param {type:"boolean"}

import os
import signal
import socket
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display

PORT = 8004
PROJECT = Path(Path("/content/softmeta_project_path.txt").read_text().strip())
LOG = Path("/content/softmeta_chatterbox.log")
PID_FILE = Path("/content/softmeta_chatterbox.pid")
MM = "/content/bin/micromamba"


def port_open() -> bool:
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=0.5):
            return True
    except OSError:
        return False


def terminate_previous() -> None:
    if PID_FILE.exists():
        try:
            old_pid = int(PID_FILE.read_text().strip())
            os.killpg(os.getpgid(old_pid), signal.SIGTERM)
            time.sleep(0.7)
        except Exception:
            pass
        PID_FILE.unlink(missing_ok=True)
    subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)


def launch_server():
    env = {
        **os.environ,
        "PYTHONUNBUFFERED": "1",
        "MPLBACKEND": "Agg",
        "HF_HOME": "/content/hf_home",
        "HF_HUB_CACHE": "/content/hf_home/hub",
        "TRANSFORMERS_CACHE": "/content/hf_home/transformers",
        "SOFTMETA_DEVICE": "cuda",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128",
        "SOFTMETA_MODEL": "chatterbox-turbo",
    }
    Path(env["HF_HOME"]).mkdir(parents=True, exist_ok=True)
    handle = LOG.open("a", encoding="utf-8", errors="replace")
    proc = subprocess.Popen(
        [MM, "run", "-n", "sm311", "python", "-u", "start.py"],
        cwd=PROJECT,
        stdout=handle,
        stderr=subprocess.STDOUT,
        env=env,
        start_new_session=True,
    )
    PID_FILE.write_text(str(proc.pid), encoding="utf-8")
    for _ in range(300):
        if proc.poll() is not None:
            handle.flush()
            tail = LOG.read_text(errors="replace")[-20000:] if LOG.exists() else ""
            handle.close()
            raise RuntimeError(tail or f"Server exited with code {proc.returncode}")
        if port_open():
            return proc, handle
        time.sleep(1)
    handle.close()
    raise TimeoutError("The server did not open port 8004. Check /content/softmeta_chatterbox.log")


terminate_previous()
LOG.unlink(missing_ok=True)
print("Starting SoftMeta Chatterbox Turbo...")
process, log_handle = launch_server()

from urllib.request import urlopen
with urlopen(f"http://127.0.0.1:{PORT}/", timeout=15) as response:
    page_status = response.status
    page_preview = response.read(300).decode("utf-8", errors="replace")
if page_status != 200 or "<html" not in page_preview.lower():
    raise RuntimeError(f"Unexpected home-page response. HTTP {page_status}: {page_preview}")

from google.colab.output import eval_js
url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
display(HTML(
    '<p><a href="' + url + '" target="_blank" '
    'style="display:inline-block;padding:13px 19px;background:#5f52e8;color:#fff;'
    'border-radius:8px;text-decoration:none;font-weight:700">'
    'Open SoftMeta Audio Studio</a></p>'
))
print("Home page check: HTTP 200 OK")
print("Server PID:", process.pid)
print("Server log:", LOG)

if not KEEP_RUNTIME_ACTIVE:
    log_handle.close()
    print("Background server started. Keep-alive supervisor is disabled for this run.")
else:
    print("Keep-alive supervisor is active. Leave this cell running while you use the Audio Studio.")
    try:
        while True:
            time.sleep(5)
            if process.poll() is None:
                continue
            try:
                log_handle.close()
            except Exception:
                pass
            print(f"Server exited unexpectedly with code {process.returncode}; restarting...")
            time.sleep(2)
            subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)
            process, log_handle = launch_server()
            print("Server restarted. Existing Audio Studio URL remains on port 8004.")
    except KeyboardInterrupt:
        print("Keep-alive stopped by notebook user.")
    finally:
        try:
            if process.poll() is None:
                os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        except Exception:
            pass
        try:
            log_handle.close()
        except Exception:
            pass
        PID_FILE.unlink(missing_ok=True)
        subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)
        print("SoftMeta server stopped. The Colab runtime itself remains assigned unless you disconnect it.")
